# ECBM và GA-ECBM trên Derm7pt

**Số liệu cho bài báo** · ResNet101 · 6 seed (42 / 123 / 2026 / 11 / 222 / 3333) · validation + test

Notebook train lại từ đầu hai mô hình và chấm cả hai theo **cùng một quy trình**:

| Ký hiệu | Mô hình | Nguồn |
|---|---|---|
| **A** | ECBM (baseline) | công thức của notebook `ecbm-baseline-derm7pt` |
| **D** | GA-ECBM (Gated Additive ECBM) | hướng 1, code của `ga-ecbm-ver1` |

## Tổng quan

### Chỉ số chẩn đoán
- **AUC** tính trên xác suất melanoma, không cần ngưỡng: đây là chỉ số chẩn đoán chính.
- **Acc, BAcc, Macro F1, Sens, Spec** cần ngưỡng. Ngưỡng chọn trên **toàn bộ validation** (tối đa Macro F1), riêng cho từng số nhóm được sửa k, rồi **áp nguyên vào test**. Số trên validation dùng ngưỡng chọn chéo giữa hai nửa validation.

### Can thiệp concept
- Thử **mọi tập con** của 7 nhóm concept (127 tập, k = 1…7); nhóm được sửa nhận giá trị thật.
- Báo F1, AUC, tỉ lệ **giúp** (sai → đúng) và **gây hại** (đúng → sai) so với lúc chưa sửa, và độ chính xác của concept chưa sửa.
- ECBM được chấm theo **hai cách can thiệp**:
  - *A (1 : 1 : 0.01)*: giữ cả ba năng lượng như lúc suy luận, cực tiểu chính xác trên các tổ hợp còn lại;
  - *A (giao thức tác giả)*: như `GradientInference.py` của xmed-lab/ECBM, vòng 2 chỉ dùng E_global (trọng số 0 : 0 : 3); concept chưa sửa giữ giá trị vòng 1 (trong code tác giả, E_global chọn concept bằng ngưỡng cứng nên concept chưa sửa không nhận gradient).

### Suy luận ECBM
Cực tiểu **chính xác** hàm năng lượng 1 : 1 : 0.01 bằng cách duyệt hết 8.640 tổ hợp concept. Đây là nghiệm đúng của bài toán mà Algorithm 1 của tác giả giải xấp xỉ bằng gradient.

### Kiểm định thống kê
- DeLong ghép cặp cho AUC (D so với A, từng seed, trên test).
- t-test ghép cặp theo seed cho các chỉ số chính.

### Khác với cấu hình gốc của tác giả
Cần ghi trong bài:
- concept nhiều trạng thái (7 nhóm, 28 trạng thái) thay vì nhị phân;
- ảnh 224 thay vì 299; batch 32 thay vì 64; hidden 32 thay vì 512;
- dừng sớm theo val loss (patience 15, tối đa 100 epoch) thay vì chạy đủ số epoch.

### Cài đặt trên Kaggle
- Input `ehehee/derm7pt-private`; GPU T4; Internet On; khoảng 1,5–2 giờ.
- Nếu gắn Output của chính notebook này (lần chạy trước) làm Input, checkpoint có sẵn được nạp lại, không train lại.

## 1. Cấu hình
Siêu tham số giống notebook baseline, danh sách 6 seed, thư mục lưu kết quả và cờ `EVALUATE_TEST`.

In [ ]:
# ===== 1. CẤU HÌNH =====
import os, sys, json, time, shutil, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Thiết lập danh sách 3 Random Seeds để đảm bảo tính tái lập (Reproducibility & Robustness)
SEEDS = [42, 123, 2026, 11, 222, 3333]

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

# Khởi tạo seed mặc định đầu tiên
set_seed(SEEDS[0])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Thiết bị tính toán: {DEVICE}")
print(f"Danh sách Random Seeds sẽ đánh giá: {SEEDS}")

# Siêu tham số chuẩn gốc theo bài báo (ICLR 2024, configs/cub.json)
EPOCHS = 100               # Số epoch tối đa theo bài báo
PATIENCE = 15              # Dừng sớm nếu val_loss không giảm sau 15 epoch (hoặc đặt None / 100 để chạy đủ 100 epoch)
BATCH_SIZE = 32            # Batch size chuẩn cho Derm7pt (413 mẫu train)
LR = 0.01                  # Learning rate của SGD
MOMENTUM = 0.9             # Momentum của SGD
WEIGHT_DECAY = 4e-5        # Weight decay chuẩn gốc
TRAIN_ENERGY_WEIGHTS = (3.0, 1.0, 1.0)      # lambda_xy=3, lambda_xc=1, lambda_cy=1
INFERENCE_ENERGY_WEIGHTS = (1.0, 1.0, 0.01)  # lambda_xy=1, lambda_xc=1, lambda_cy=0.01

# Backbone mạng: 'resnet50' hoặc 'resnet101' (chuẩn bài báo gốc)
BACKBONE_NAME = "resnet101" # Cả hai đều có feature dim 2048
HIDDEN_DIM = 32            # Chiều ẩn cho embedding và projection (chuẩn gốc ECBM)
DROPOUT = 0.2

# Thư mục làm việc & xuất kết quả
default_work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.home() / "Desktop/HK1_26-27/KLTN/DermaXAI-Explainable-AI-for-Skin-Lesion-Classification/results/local"
if not default_work.parent.exists():
    default_work = Path.cwd() / "results"
WORK = Path(os.environ.get("ECBM_WORK", default_work))
OUT = WORK / "ecbm_derm7pt_paper"
OUT.mkdir(parents=True, exist_ok=True)
print(f"Kết quả và Checkpoint sẽ lưu tại: {OUT}")
EVALUATE_TEST = True     # user đồng ý chấm test cho lần chạy này (12/11/2026)

## 2. Dữ liệu Derm7pt
Nạp split chính thức (train / val / test = 413 / 203 / 395 ca; melanoma 90 / 61 / 101) và 7 nhóm concept. Có `assert` để chắc chắn đúng split chính thức.

In [ ]:
# ===== 2. DỮ LIỆU DERM7PT =====

CONCEPT_NAMES = [
    "pigment_network",
    "streaks",
    "pigmentation",
    "regression_structures",
    "dots_and_globules",
    "blue_whitish_veil",
    "vascular_structures",
]

CONCEPT_NUM_CLASSES = {
    "pigment_network": 3,
    "streaks": 3,
    "pigmentation": 5,
    "regression_structures": 4,
    "dots_and_globules": 3,
    "blue_whitish_veil": 2,
    "vascular_structures": 8,
}

LABEL_MAPPING = {
    "pigment_network": {"absent": 0, "atypical": 1, "typical": 2},
    "streaks": {"absent": 0, "irregular": 1, "regular": 2},
    "pigmentation": {"absent": 0, "diffuse irregular": 1, "diffuse regular": 2, "localized irregular": 3, "localized regular": 4},
    "regression_structures": {"absent": 0, "blue areas": 1, "combinations": 2, "white areas": 3},
    "dots_and_globules": {"absent": 0, "irregular": 1, "regular": 2},
    "blue_whitish_veil": {"absent": 0, "present": 1},
    "vascular_structures": {"absent": 0, "arborizing": 1, "comma": 2, "dotted": 3, "hairpin": 4, "linear irregular": 5, "within regression": 6, "wreath": 7}
}

def load_derm7pt_dataset():
    # Các thư mục tìm kiếm tiềm năng
    search_dirs = []
    if Path("/kaggle/input").exists():
        search_dirs.append(Path("/kaggle/input"))
    local_repo = Path.home() / "Desktop/HK1_26-27/KLTN/DermaXAI-Explainable-AI-for-Skin-Lesion-Classification"
    if local_repo.exists():
        search_dirs.append(local_repo)
    for p in [Path.cwd() / "data", Path.cwd() / "Derm7pt", Path.cwd()]:
        if p.exists() and p not in search_dirs:
            search_dirs.append(p)
            
    print(f"Đang tìm kiếm dữ liệu Derm7pt trong: {[str(d) for d in search_dirs]}...")

    # 1. Tìm manifest.csv nếu có
    for sdir in search_dirs:
        for hit in sdir.rglob("manifest.csv"):
            print(f"✓ Tìm thấy manifest.csv tại: {hit}")
            df = pd.read_csv(hit)
            img_dir = None
            for cand in [hit.parent / "images", hit.parent.parent / "Derm7pt/release_v0/images", hit.parent]:
                if cand.exists() and cand.is_dir():
                    img_dir = cand
                    break
            image_map = {}
            if img_dir:
                for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
                    for f in img_dir.rglob(ext):
                        image_map[f.name.lower()] = f
            return df, image_map

    # 2. Tìm trong dataset raw Derm7pt (chứa meta/meta.csv hoặc meta*.csv)
    for sdir in search_dirs:
        for hit in sdir.rglob("*meta*.csv"):
            if "manifest" in hit.name: continue
            try:
                meta_df = pd.read_csv(hit)
            except Exception:
                continue
            if not ("diagnosis" in meta_df.columns or "seven_point_score" in meta_df.columns or "derm" in meta_df.columns):
                continue

            print(f"✓ Tìm thấy file metadata Derm7pt tại: {hit}")
            
            # Tìm các file train_indexes.csv, valid_indexes.csv, test_indexes.csv
            meta_dir = hit.parent
            tr_file, va_file, te_file = None, None, None
            for candidate_dir in [meta_dir, meta_dir.parent, meta_dir / "meta"]:
                if (candidate_dir / "train_indexes.csv").exists():
                    tr_file = candidate_dir / "train_indexes.csv"
                    va_file = candidate_dir / "valid_indexes.csv"
                    te_file = candidate_dir / "test_indexes.csv"
                    break
            
            if tr_file and tr_file.exists() and va_file and va_file.exists() and te_file and te_file.exists():
                print(f"✓ Tìm thấy đầy đủ file phân chia index: {tr_file.parent}")
                tr_idx = pd.read_csv(tr_file)["indexes"]
                va_idx = pd.read_csv(va_file)["indexes"]
                te_idx = pd.read_csv(te_file)["indexes"]
                meta_df["split"] = "none"
                meta_df.loc[tr_idx, "split"] = "train"
                meta_df.loc[va_idx, "split"] = "valid"
                meta_df.loc[te_idx, "split"] = "test"
                df = meta_df[meta_df["split"] != "none"].copy()
            elif "split" in meta_df.columns:
                print("✓ Sử dụng cột split sẵn có trong metadata.")
                df = meta_df.copy()
            else:
                print("! Không tìm thấy index csv, tự động phân chia train/val/test (60%/20%/20%).")
                np.random.seed(42)
                shuffled_idx = np.random.permutation(len(meta_df))
                n_tr = int(len(meta_df) * 0.6)
                n_va = int(len(meta_df) * 0.2)
                meta_df["split"] = "none"
                meta_df.iloc[shuffled_idx[:n_tr], meta_df.columns.get_loc("split")] = "train"
                meta_df.iloc[shuffled_idx[n_tr:n_tr+n_va], meta_df.columns.get_loc("split")] = "valid"
                meta_df.iloc[shuffled_idx[n_tr+n_va:], meta_df.columns.get_loc("split")] = "test"
                df = meta_df.copy()

            if "diagnosis_binary" not in df.columns:
                df["diagnosis_binary"] = df["diagnosis"].apply(lambda d: 1 if "mel" in str(d).lower() else 0)

            # Tìm thư mục images: quét gần vị trí metadata trước
            img_dir = None
            for cand in [meta_dir.parent / "images", meta_dir / "images", hit.parent.parent.parent / "images"]:
                if cand.exists() and cand.is_dir():
                    img_dir = cand
                    break
            if not img_dir:
                for cand in sdir.rglob("images"):
                    if cand.is_dir():
                        img_dir = cand
                        break

            image_map = {}
            if img_dir and img_dir.exists():
                print(f"✓ Đang lập chỉ mục ảnh từ thư mục: {img_dir}")
                for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
                    for f in img_dir.rglob(ext):
                        image_map[f.name.lower()] = f
                print(f"✓ Đã lập chỉ mục {len(image_map)} ảnh.")

            return df, image_map

    # Nếu không tìm thấy, in ra danh sách thư mục /kaggle/input để người dùng dễ kiểm tra
    print("\n--- DANH SÁCH FILE TRONG MÔI TRƯỜNG HIỆN TẠI ---")
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob("*"):
            if p.is_dir() or p.suffix in [".csv", ".txt", ".json"]:
                print(" -", p)
    return None, None

df_all, IMAGE_MAP = load_derm7pt_dataset()
assert df_all is not None, (
    "LỖI: Không tìm thấy dữ liệu Derm7pt trong /kaggle/input hoặc thư mục máy! "
    "Vui lòng bấm '+ Add Input' trên Kaggle và thêm bộ dữ liệu Derm7pt (chứa thư mục images và meta)."
)
# SỬA LỖI NHÃN so với baseline cũ: "mel" in diagnosis bắt cả 16 ca "melanosis" (lành tính).
# Nhãn đúng: melanoma = chẩn đoán bắt đầu bằng "melanoma" (nhóm MEL của Kawahara et al. 2019).
df_all["diagnosis_binary"] = df_all["diagnosis"].astype(str).str.lower().str.startswith("melanoma").astype(int)

# Thống kê tập dữ liệu
train_df = df_all[df_all["split"] == "train"].copy()
val_df   = df_all[df_all["split"].isin(["val", "valid"])].copy()
test_df  = df_all[df_all["split"] == "test"].copy()
print(f"✓ Nạp dữ liệu thành công! Tổng {len(df_all)} ca:")
print(f"  - Train     : {len(train_df)} ca (Melanoma: {train_df['diagnosis_binary'].sum()})")
print(f"  - Validation: {len(val_df)} ca (Melanoma: {val_df['diagnosis_binary'].sum()})")
print(f"  - Test      : {len(test_df)} ca (Melanoma: {test_df['diagnosis_binary'].sum()})")

# Image transforms (Resize chuẩn 224x224 để đồng nhất kích thước batch)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class Derm7ptECBMDataset(Dataset):
    def __init__(self, df, transform=None, image_map=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.image_map = image_map or {}

    def __len__(self):
        return len(self.df)

    def _resolve_img_path(self, row):
        if "derm_path_resolved" in row and os.path.exists(str(row["derm_path_resolved"])):
            return Path(row["derm_path_resolved"])
        
        derm_rel = str(row["derm"]).strip() if "derm" in row and pd.notna(row["derm"]) else ""
        fname = Path(derm_rel).name.lower()
        
        if fname in self.image_map:
            return self.image_map[fname]
            
        for root in [Path("/kaggle/input"), Path(".")]:
            cand = root / derm_rel
            if cand.exists(): return cand
            
        return Path(derm_rel)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._resolve_img_path(row)
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (224, 224), color=(128, 128, 128))

        if self.transform:
            image = self.transform(image)

        y = int(row["diagnosis_binary"])
        concepts = []
        for name in CONCEPT_NAMES:
            raw_val = row[name] if name in row and pd.notna(row[name]) else "absent"
            val_idx = LABEL_MAPPING[name].get(str(raw_val).strip().lower(), 0)
            concepts.append(val_idx)

        return image, torch.tensor(y, dtype=torch.long), torch.tensor(concepts, dtype=torch.long)

def get_dataloaders(seed=42):
    """Khởi tạo DataLoader với generator theo seed cụ thể để đảm bảo tái lập thứ tự shuffle."""
    g = torch.Generator()
    g.manual_seed(seed)
    tr_loader = DataLoader(Derm7ptECBMDataset(train_df, train_transform, IMAGE_MAP), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, generator=g)
    va_loader = DataLoader(Derm7ptECBMDataset(val_df, eval_transform, IMAGE_MAP), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    te_loader = DataLoader(Derm7ptECBMDataset(test_df, eval_transform, IMAGE_MAP), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return tr_loader, va_loader, te_loader

train_loader, val_loader, test_loader = get_dataloaders(SEEDS[0])
print(f"✓ Đã khởi tạo xong DataLoaders (Train: {len(train_loader)} batches, Val: {len(val_loader)}, Test: {len(test_loader)})!")

EXPECTED = {"train": (413, 90), "val": (203, 61), "test": (395, 101)}   # (số ca, số melanoma) theo split chính thức
for name, frame in (("train", train_df), ("val", val_df), ("test", test_df)):
    if len(frame) == EXPECTED[name][0]:
        assert int(frame["diagnosis_binary"].sum()) == EXPECTED[name][1], f"{name}: sai số melanoma"
print("✓ Nhãn melanoma đúng: train/val/test =", [int(f["diagnosis_binary"].sum()) for f in (train_df, val_df, test_df)])

for name, frame in (("train", train_df), ("val", val_df), ("test", test_df)):     # bắt buộc đúng split chính thức
    assert (len(frame), int(frame["diagnosis_binary"].sum())) == EXPECTED[name], f"{name}: không phải split chính thức"
print("✓ Đúng split chính thức Derm7pt (413/203/395; melanoma 90/61/101)")

## 3. Mô hình ECBM (A)
Backbone ResNet101 và ba hàm năng lượng: E_class (ảnh → nhãn), E_concept (ảnh → concept), E_global (concept → nhãn).

In [ ]:
# ===== 3. MÔ HÌNH ECBM (A) =====

def _gate(hidden: torch.Tensor, embedding: torch.Tensor) -> torch.Tensor:
    """Cổng kích hoạt kết hợp đặc trưng và embedding chuẩn theo mã nguồn tác giả."""
    return F.relu(hidden * (1.0 + F.normalize(embedding, dim=-1, eps=1e-12)))

class CategoricalECBM(nn.Module):
    def __init__(self, backbone_name="resnet101", num_classes=2, hidden_dim=32, dropout=0.2):
        super().__init__()
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        self.concept_names = CONCEPT_NAMES
        self.state_counts = [CONCEPT_NUM_CLASSES[name] for name in self.concept_names]

        # 1. Backbone CNN trích xuất đặc trưng (2048 chiều)
        if backbone_name == "resnet50":
            weights = models.ResNet50_Weights.IMAGENET1K_V2
            backbone = models.resnet50(weights=weights)
            in_features = backbone.fc.in_features
            self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        elif backbone_name == "resnet101":
            weights = models.ResNet101_Weights.IMAGENET1K_V2
            backbone = models.resnet101(weights=weights)
            in_features = backbone.fc.in_features
            self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        else:
            raise ValueError(f"Không hỗ trợ backbone: {backbone_name}")

        self.dropout = nn.Dropout(p=dropout)

        # 2. Embedding bảng nhãn và các trạng thái khái niệm
        self.class_embeddings = nn.Parameter(torch.randn(num_classes, hidden_dim))
        self.concept_embeddings = nn.ParameterList([
            nn.Parameter(torch.randn(count, hidden_dim)) for count in self.state_counts
        ])

        # 3. Các tầng chiếu tuyến tính cho 3 hàm năng lượng
        # E_class
        self.class_proj = nn.Linear(in_features, hidden_dim)
        self.class_out = nn.Linear(hidden_dim, 1)

        # E_concept
        self.concept_proj = nn.Linear(in_features, hidden_dim)
        self.concept_outs = nn.ModuleList([nn.Linear(hidden_dim, 1) for _ in self.state_counts])

        # E_global
        self.global_proj = nn.Linear(len(self.state_counts) * hidden_dim, hidden_dim)
        self.global_out = nn.Linear(hidden_dim, 1)

        # Lưu trữ toàn bộ 8.640 tổ hợp concept cho Exact Inference
        all_states = [list(range(c)) for c in self.state_counts]
        import itertools
        profiles = torch.tensor(list(itertools.product(*all_states)), dtype=torch.long)
        self.register_buffer("profiles", profiles, persistent=False)

    def extract_features(self, x):
        feat = self.backbone(x)
        feat = torch.flatten(feat, 1)
        return feat

    def energy_class(self, feat):
        """Tính năng lượng E_class(x, y) cho tất cả các class y."""
        h = self.dropout(F.relu(self.class_proj(feat)))
        energies = []
        for y in range(self.num_classes):
            e_y = self.class_embeddings[y]
            gated = _gate(h, e_y.unsqueeze(0))
            energy = self.class_out(gated)
            energies.append(energy)
        return torch.cat(energies, dim=1)

    def energy_concept(self, feat):
        """Tính năng lượng E_concept(x, c_g) cho từng nhóm khái niệm."""
        h = self.dropout(F.relu(self.concept_proj(feat)))
        energies_per_group = []
        for g, count in enumerate(self.state_counts):
            group_energies = []
            for s in range(count):
                e_s = self.concept_embeddings[g][s]
                gated = _gate(h, e_s.unsqueeze(0))
                energy = self.concept_outs[g](gated)
                group_energies.append(energy)
            energies_per_group.append(torch.cat(group_energies, dim=1))
        return energies_per_group

    def energy_global(self, concepts_idx):
        """Tính năng lượng E_global(c, y) giữa vector khái niệm và nhãn y."""
        c_embs = []
        for g in range(len(self.state_counts)):
            emb = self.concept_embeddings[g][concepts_idx[:, g]]
            c_embs.append(emb)
        concat_emb = torch.cat(c_embs, dim=1)
        h = self.dropout(F.relu(self.global_proj(concat_emb)))

        energies = []
        for y in range(self.num_classes):
            e_y = self.class_embeddings[y]
            gated = _gate(h, e_y.unsqueeze(0))
            energy = self.global_out(gated)
            energies.append(energy)
        return torch.cat(energies, dim=1)

model = CategoricalECBM(backbone_name=BACKBONE_NAME, num_classes=2, hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)
print(f"✓ Khởi tạo thành công mô hình ECBM với backbone {BACKBONE_NAME} (2048 dims)!")


def create_model():
    """Tạo một phiên bản mới của mô hình Categorical ECBM theo đúng kiến trúc chuẩn."""
    return CategoricalECBM(backbone_name=BACKBONE_NAME, num_classes=2, hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)

model = create_model()
print(f"✓ Đã định nghĩa kiến trúc Categorical ECBM ({BACKBONE_NAME}) trên {DEVICE}!")

## 4. Loss của ECBM
3·L_xy + L_xc + L_cy, xáo trộn concept khi tính L_cy như code xmed-lab/ECBM. Hàm này cũng là **val loss để chọn checkpoint cho cả A và D**.

In [ ]:
# ===== 4. LOSS CỦA ECBM =====
def perturb_concepts(model, c_true, perturb_prob=0.2, permute_prob=0.2):
    """Thêm nhiễu ngẫu nhiên vào vector concept chuẩn theo code xmed-lab/ECBM."""
    c_noisy = c_true.clone()
    B, G = c_noisy.shape
    # 1. Đổi ngẫu nhiên một số concept sang trạng thái khác
    mask_perturb = (torch.rand_like(c_noisy.float()) < perturb_prob)
    for g, count in enumerate(model.state_counts):
        random_states = torch.randint(0, count, (B,), device=c_true.device)
        c_noisy[:, g] = torch.where(mask_perturb[:, g], random_states, c_noisy[:, g])
    # 2. Tráo đổi concept giữa các mẫu
    if torch.rand(1).item() < permute_prob:
        perm = torch.randperm(B)
        c_noisy = c_noisy[perm]
    return c_noisy

def compute_loss(model, images, y_true, c_true):
    feat = model.extract_features(images)
    
    # 1. L_xy (Ảnh -> Nhãn)
    e_xy = model.energy_class(feat)
    loss_xy = F.cross_entropy(-e_xy, y_true)
    
    # 2. L_xc (Ảnh -> Khái niệm)
    e_xc_list = model.energy_concept(feat)
    loss_xc = 0.0
    for g, e_g in enumerate(e_xc_list):
        loss_xc += F.cross_entropy(-e_g, c_true[:, g])
        
    # 3. L_cy (Khái niệm -> Nhãn)
    c_perturbed = perturb_concepts(model, c_true, 0.2, 0.2)
    e_cy = model.energy_global(c_perturbed)
    loss_cy = F.cross_entropy(-e_cy, y_true)
    
    # Tổng loss theo trọng số chuẩn: 3 * L_xy + 1 * L_xc + 1 * L_cy
    w_xy, w_xc, w_cy = TRAIN_ENERGY_WEIGHTS
    total_loss = w_xy * loss_xy + w_xc * loss_xc + w_cy * loss_cy
    return total_loss, loss_xy.item(), loss_xc.item(), loss_cy.item()

## 5. Suy luận chính xác
Duyệt hết 8.640 tổ hợp concept để tìm cực tiểu của năng lượng 1 : 1 : 0.01.

In [ ]:
# ===== 5. SUY LUẬN CHÍNH XÁC =====
def exact_inference(model, feat):
    """Suy luận chính xác (Exact Enumeration) qua toàn bộ 8.640 tổ hợp concept."""
    B = feat.shape[0]
    w_xy, w_xc, w_cy = INFERENCE_ENERGY_WEIGHTS
    
    # 1. E_class(x, y): (B, 2)
    e_xy = model.energy_class(feat)
    
    # 2. E_concept(x, c): Tính trước bảng năng lượng cho từng trạng thái
    e_xc_list = model.energy_concept(feat)
    
    # 3. Quét qua toàn bộ profiles: (8640, 7)
    profiles = model.profiles.to(DEVICE)
    N_p = profiles.shape[0]
    
    e_xc_profiles = torch.zeros(B, N_p, device=DEVICE)
    for g in range(len(model.state_counts)):
        e_g = e_xc_list[g]
        p_g = profiles[:, g]
        e_xc_profiles += e_g[:, p_g]
        
    e_cy_profiles = model.energy_global(profiles)
    
    best_energies = []
    best_c_preds = []
    for y in range(model.num_classes):
        total_e = w_xy * e_xy[:, y].unsqueeze(1) + w_xc * e_xc_profiles + w_cy * e_cy_profiles[:, y].unsqueeze(0)
        min_e, min_idx = total_e.min(dim=1)
        best_energies.append(min_e.unsqueeze(1))
        best_c_preds.append(profiles[min_idx])
        
    best_energies = torch.cat(best_energies, dim=1)
    pred_y = (-best_energies).argmax(dim=1)
    
    final_c = torch.zeros(B, len(model.state_counts), dtype=torch.long, device=DEVICE)
    for b in range(B):
        final_c[b] = best_c_preds[pred_y[b]][b]
        
    probs_y = F.softmax(-best_energies, dim=1)[:, 1]
    return pred_y, probs_y, final_c


@torch.no_grad()
def baseline_scores(model, feat, profile_energy_cache=None, mask=None, values=None):
    # exact_inference của baseline, mở rộng cho can thiệp: chỉ giữ tổ hợp khớp concept đã xác minh.
    w_xy, w_xc, w_cy = INFERENCE_ENERGY_WEIGHTS
    e_xy = model.energy_class(feat)
    profiles = model.profiles.to(DEVICE)
    e_xc = sum(e_g[:, profiles[:, g]] for g, e_g in enumerate(model.energy_concept(feat)))     # [B, P]
    e_cy = model.energy_global(profiles) if profile_energy_cache is None else profile_energy_cache  # [P, 2]
    if mask is not None:
        allowed = ((profiles[None] == values[:, None, :]) | ~mask[:, None, :]).all(-1)           # [B, P]
        e_xc = e_xc.masked_fill(~allowed, float("inf"))
    best, best_idx = [], []
    for y in range(model.num_classes):
        total = w_xy * e_xy[:, y:y + 1] + w_xc * e_xc + w_cy * e_cy[:, y][None]
        energy, index = total.min(dim=1)
        best.append(energy); best_idx.append(index)
    best = torch.stack(best, 1)
    pred_y = (-best).argmax(1)
    chosen = torch.stack(best_idx, 1).gather(1, pred_y[:, None]).squeeze(1)
    return F.softmax(-best, dim=1)[:, 1], profiles[chosen]

## 6. Mô hình GA-ECBM (D)
- E_global thay bằng bảng cộng tính: E_global(c, y) = −(b_y + Σ_g θ_g(y, c_g)).
- Nhờ cộng tính, posterior p(y | x, concept đã sửa) tính được **dạng đóng**, không cần tối ưu.
- Cổng w(k) giảm dần theo số nhóm đã sửa k, để nhánh ảnh nhường dần cho concept.
- Loss = loss ECBM (mục 4) + loss posterior + loss can thiệp + loss đơn điệu.

In [ ]:
# ===== 6. MÔ HÌNH GA-ECBM (D) =====
class GatedAdditiveECBM(CategoricalECBM):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        del self.global_proj, self.global_out                   # bỏ MLP của E_global
        self.theta = nn.ParameterList([nn.Parameter(torch.zeros(self.num_classes, k)) for k in self.state_counts])
        self.label_bias = nn.Parameter(torch.zeros(self.num_classes))
        self.gate_steps = nn.Parameter(torch.full((len(self.state_counts),), -1.5))

    def energy_global(self, concepts_idx):
        # E_global(c, y) = −(b_y + Σ_g θ_{y,g}[c_g]); dùng được cho loss L_cy của baseline.
        score = self.label_bias[None].expand(len(concepts_idx), -1)
        for g, theta in enumerate(self.theta):
            score = score + theta[:, concepts_idx[:, g]].t()
        return -score

    def gate(self, verified_groups):
        w = torch.exp(-torch.cat([self.gate_steps.new_zeros(1), torch.cumsum(F.softplus(self.gate_steps), 0)]))
        return w[verified_groups]                                 # w(0) = 1, giảm dần theo k

    def posterior(self, feat, mask=None, values=None, class_weight=None):
        # log p(y | x, concept đã biết) và p(c_g = s | x, …): chính xác, vì E_global cộng tính theo từng nhóm.
        e_xy = self.energy_class(feat)
        weight = 1.0 if class_weight is None else class_weight[:, None]
        logits = -weight * e_xy + self.label_bias[None]
        terms = []
        for g, e_g in enumerate(self.energy_concept(feat)):
            t = -e_g[:, None, :] + self.theta[g][None]            # [B, 2, K_g]
            if mask is not None:
                keep = (~mask[:, g:g + 1]) | (torch.arange(t.shape[-1], device=t.device)[None] == values[:, g:g + 1])
                t = t.masked_fill(~keep[:, None, :], -1e4)
            terms.append(t)
            logits = logits + torch.logsumexp(t, -1)
        log_p_y = logits.log_softmax(-1)
        p_concepts = [(log_p_y.exp()[:, :, None] * t.softmax(-1)).sum(1) for t in terms]
        return log_p_y, p_concepts


def random_group_mask(batch, groups, device):
    ratio = torch.rand(batch, 1, device=device)
    return torch.rand(batch, groups, device=device) < ratio


def baseline_loss_on_features(model, feat, y_true, c_true):
    # Đúng công thức compute_loss của baseline (3·L_xy + L_xc + L_cy, xáo trộn concept như baseline),
    # chỉ khác là nhận đặc trưng đã tính sẵn để backbone không phải chạy hai lần mỗi batch.
    loss_xy = F.cross_entropy(-model.energy_class(feat), y_true)
    loss_xc = sum(F.cross_entropy(-e_g, c_true[:, g]) for g, e_g in enumerate(model.energy_concept(feat)))
    loss_cy = F.cross_entropy(-model.energy_global(perturb_concepts(model, c_true, 0.2, 0.2)), y_true)
    w_xy, w_xc, w_cy = TRAIN_ENERGY_WEIGHTS
    return w_xy * loss_xy + w_xc * loss_xc + w_cy * loss_cy


def gated_loss(model, images, y_true, c_true):
    # Loss gốc của baseline (3 : 1 : 1) + loss biên + loss can thiệp có cổng + loss đơn điệu.
    feat = model.extract_features(images)
    base = baseline_loss_on_features(model, feat, y_true, c_true)
    log_p_y, p_c = model.posterior(feat)
    joint = F.nll_loss(log_p_y, y_true) + sum(F.nll_loss(torch.log(p.clamp_min(1e-8)), c_true[:, g])
                                              for g, p in enumerate(p_c)) / len(p_c)
    groups = len(model.state_counts)
    mask = random_group_mask(len(y_true), groups, images.device)
    drop = torch.rand(len(y_true), device=images.device) < 0.5            # nửa số ca tắt hẳn nhánh ảnh
    k = mask.sum(1)
    w_k = torch.where(drop, torch.zeros_like(k, dtype=torch.float), model.gate(k))
    log_int, _ = model.posterior(feat, mask, c_true, w_k)
    loss_int = F.nll_loss(log_int, y_true, reduction="none")
    extra = torch.multinomial((~mask).float() + 1e-9, 1).squeeze(1)       # xác minh thêm 1 nhóm
    mask_more = mask.clone()
    mask_more[torch.arange(len(y_true)), extra] = True
    k_more = mask_more.sum(1)
    w_more = torch.where(drop, torch.zeros_like(k_more, dtype=torch.float), model.gate(k_more))
    log_more, _ = model.posterior(feat, mask_more, c_true, w_more)
    mono = F.relu(F.nll_loss(log_more, y_true, reduction="none") - loss_int).mean()
    return base + joint + loss_int.mean() + mono


def create_gated_model():
    return GatedAdditiveECBM(backbone_name=BACKBONE_NAME, num_classes=2, hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)

print("✓ Đã định nghĩa Gated Additive ECBM")

## 7. Vòng train dùng chung
A và D dùng cùng vòng train: SGD, ReduceLROnPlateau, dừng sớm (patience 15, tối đa 100 epoch). Checkpoint chọn theo val loss của mục 4 cho cả hai mô hình.

In [ ]:
# ===== 7. VÒNG TRAIN DÙNG CHUNG =====
GATE_LR = 100 * LR

def default_optimizer(m):
    return torch.optim.SGD(m.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

def gated_optimizer(m):
    # Thay đổi duy nhất của #1b ở phần train: cổng w(k) có learning rate riêng để học được thật.
    rest = [p for name, p in m.named_parameters() if name != "gate_steps"]
    return torch.optim.SGD([{"params": rest}, {"params": [m.gate_steps], "lr": GATE_LR}],
                           lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

def train_model(seed, make_model, train_loss_fn, tag, make_optimizer=default_optimizer, epochs=EPOCHS, patience=PATIENCE):
    set_seed(seed)
    m = make_model()
    tr_loader, va_loader, _ = get_dataloaders(seed)
    opt = make_optimizer(m)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.1, patience=10)
    best_loss, no_improve, hist = float("inf"), 0, []
    ckpt = OUT / f"{tag}_seed_{seed}.pth"
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        m.train()
        total = 0.0
        for imgs, y_b, c_b in tr_loader:
            imgs, y_b, c_b = imgs.to(DEVICE), y_b.to(DEVICE), c_b.to(DEVICE)
            opt.zero_grad()
            loss = train_loss_fn(m, imgs, y_b, c_b)
            loss = loss[0] if isinstance(loss, tuple) else loss
            loss.backward()
            opt.step()
            total += loss.item() * len(imgs)
        train_loss = total / len(tr_loader.dataset)
        m.eval()
        val_total = 0.0
        with torch.no_grad():
            for imgs, y_b, c_b in va_loader:
                imgs, y_b, c_b = imgs.to(DEVICE), y_b.to(DEVICE), c_b.to(DEVICE)
                val_total += compute_loss(m, imgs, y_b, c_b)[0].item() * len(imgs)
        val_loss = val_total / len(va_loader.dataset)
        sched.step(train_loss)
        hist.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        if val_loss < best_loss:
            best_loss, no_improve = val_loss, 0
            torch.save(m.state_dict(), ckpt)
        else:
            no_improve += 1
        if patience is not None and no_improve >= patience:
            break
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    print(f"  {tag} seed {seed}: {len(hist)} epoch, val loss tốt nhất {best_loss:.4f}, {(time.time() - t0) / 60:.1f} phút",
          flush=True)
    return m.eval(), hist

## 8. Hàm chấm chung
Với mỗi mô hình và mỗi tập (val / test): điểm melanoma và concept dự đoán ở k = 0 và ở cả 127 tập con can thiệp. Mọi chỉ số (ngưỡng, giúp / gây hại, DeLong) tính từ các ma trận điểm này. DeLong được đối chiếu với `sklearn` khi chạy.

In [ ]:
# ===== 8. HÀM CHẤM CHUNG =====
from itertools import combinations
from scipy import stats

GROUPS = len(CONCEPT_NAMES)
SUBSETS = [s for k in range(1, GROUPS + 1) for s in combinations(range(GROUPS), k)]   # 127 tập con
AUTHOR_ROUND2_WEIGHT = 3.0                                                            # λ_cy vòng 2 của tác giả


def best_threshold(y_true, scores):
    # Ngưỡng tối đa Macro F1 (melanoma khi score >= ngưỡng); duyệt mọi điểm cắt.
    order = np.argsort(-scores, kind="mergesort")
    s, y = scores[order], y_true[order].astype(float)
    pos, neg = y.sum(), len(y) - y.sum()
    tp, fp = np.cumsum(y), np.cumsum(1 - y)
    fn, tn = pos - tp, neg - fp
    macro = (2 * tp / np.maximum(2 * tp + fp + fn, 1e-12) + 2 * tn / np.maximum(2 * tn + fn + fp, 1e-12)) / 2
    macro = np.where(np.r_[s[1:] != s[:-1], True], macro, -1.0)
    i = int(np.argmax(macro))
    nxt = s[i + 1] if i + 1 < len(s) else s[i] - 1e-6
    return float((s[i] + nxt) / 2)


def val_halves(y_true):
    rng = np.random.RandomState(0)
    half = np.zeros(len(y_true), dtype=int)
    for label in (0, 1):
        idx = np.where(y_true == label)[0]
        rng.shuffle(idx)
        half[idx[len(idx) // 2:]] = 1
    return half


@torch.no_grad()
def collect(model, loader):
    feats, ys, cs = [], [], []
    for imgs, y_b, c_b in loader:
        feats.append(model.extract_features(imgs.to(DEVICE)))
        ys.append(y_b); cs.append(c_b)
    return torch.cat(feats), torch.cat(ys).numpy(), torch.cat(cs).to(DEVICE)


@torch.no_grad()
def score_all(variant, model, feat, c_true):
    """Điểm melanoma [1 + 127, N] và concept dự đoán [1 + 127, N, 7] cho k = 0 và mọi tập con."""
    n = len(feat)
    masks = [None]
    for subset in SUBSETS:
        m = torch.zeros(n, GROUPS, dtype=torch.bool, device=DEVICE)
        m[:, list(subset)] = True
        masks.append(m)
    scores, concepts = [], []
    if variant == "D":
        for m in masks:
            k = None if m is None else m.sum(1)
            log_p_y, p_c = model.posterior(feat, m, c_true, None if k is None else model.gate(k))
            scores.append(log_p_y.exp()[:, 1]); concepts.append(torch.stack([p.argmax(1) for p in p_c], 1))
    else:
        cache = model.energy_global(model.profiles.to(DEVICE))
        s0, c0 = baseline_scores(model, feat, cache)                       # vòng 1: cực tiểu chính xác 1 : 1 : 0.01
        scores.append(s0); concepts.append(c0)
        for m in masks[1:]:
            if variant == "A_joint":
                s, c = baseline_scores(model, feat, cache, m, c_true)
            else:                                                            # giao thức tác giả: vòng 2 chỉ E_global
                c = torch.where(m, c_true, c0)
                s = F.softmax(-AUTHOR_ROUND2_WEIGHT * model.energy_global(c), dim=1)[:, 1]
            scores.append(s); concepts.append(c)
    return torch.stack(scores).cpu().numpy(), torch.stack(concepts).cpu().numpy(), masks


def diag(y, s, p):
    return {"accuracy": accuracy_score(y, p), "balanced_accuracy": balanced_accuracy_score(y, p),
            "f1_macro": f1_score(y, p, average="macro", zero_division=0),
            "f1_melanoma": f1_score(y, p, zero_division=0), "precision": precision_score(y, p, zero_division=0),
            "sensitivity": recall_score(y, p, zero_division=0), "specificity": recall_score(1 - y, 1 - p, zero_division=0),
            "roc_auc": roc_auc_score(y, s)}


def concept_report(c_true, c_pred):
    per = {}
    for g, name in enumerate(CONCEPT_NAMES):
        per[name] = {"acc": float((c_true[:, g] == c_pred[:, g]).mean()),
                     "f1_macro": float(f1_score(c_true[:, g], c_pred[:, g], average="macro",
                                                labels=list(range(CONCEPT_NUM_CLASSES[name])), zero_division=0))}
    return {"concept_accuracy": float((c_true == c_pred).mean()),
            "concept_f1_macro": float(np.mean([v["f1_macro"] for v in per.values()])),
            "exact_match": float((c_true == c_pred).all(1).mean()), "per_concept": per}


def ks_index():
    ks = np.array([0] + [len(s) for s in SUBSETS])
    return ks


def predictions(y_val, S_val, S_eval, same_split):
    """Nhãn dự đoán [1+127, N] cho tập cần chấm; ngưỡng theo k (gộp mọi tập con cùng k) chọn trên validation."""
    ks = ks_index()
    pred = np.zeros(S_eval.shape, dtype=int)
    thresholds = {}
    half = val_halves(y_val)
    for k in range(GROUPS + 1):
        rows = np.where(ks == k)[0]
        if same_split:                                                       # validation: chọn chéo hai nửa
            ts = []
            for h in (0, 1):
                other, this = half != h, half == h
                t = best_threshold(np.tile(y_val[other], len(rows)), S_val[rows][:, other].ravel())
                pred[np.ix_(rows, np.where(this)[0])] = (S_eval[rows][:, this] >= t).astype(int)
                ts.append(t)
            thresholds[k] = ts
        else:                                                                # test: ngưỡng chọn trên toàn validation
            t = best_threshold(np.tile(y_val, len(rows)), S_val[rows].ravel())
            pred[rows] = (S_eval[rows] >= t).astype(int)
            thresholds[k] = t
    return pred, thresholds


def summarize(y, S, C, pred, c_true, masks):
    ks = ks_index()
    base = pred[0]
    out = {"unintervened": {**diag(y, S[0], base), **{k: v for k, v in concept_report(c_true, C[0]).items()}}}
    by_k = {}
    for k in range(1, GROUPS + 1):
        rows = []
        for r in np.where(ks == k)[0]:
            m = masks[r].cpu().numpy()
            unver = ~m
            rows.append({"f1_macro": f1_score(y, pred[r], average="macro", zero_division=0),
                         "balanced_accuracy": balanced_accuracy_score(y, pred[r]),
                         "roc_auc": roc_auc_score(y, S[r]),
                         "harm_rate": float(((base == y) & (pred[r] != y)).mean()),
                         "help_rate": float(((base != y) & (pred[r] == y)).mean()),
                         "unverified_concept_accuracy": float((C[r] == c_true)[unver].mean()) if unver.any() else None})
        by_k[str(k)] = {key: (None if rows[0][key] is None else float(np.mean([row[key] for row in rows]))) for key in rows[0]}
    out["intervention_by_k"] = by_k
    mk = lambda key, kk: float(np.mean([by_k[str(k)][key] for k in kk]))
    out.update({"f1_k1_3": mk("f1_macro", (1, 2, 3)), "auc_k1_3": mk("roc_auc", (1, 2, 3)),
                "harm_k1_3": mk("harm_rate", (1, 2, 3)), "help_k1_3": mk("help_rate", (1, 2, 3)),
                "f1_k7": by_k[str(GROUPS)]["f1_macro"], "auc_k7": by_k[str(GROUPS)]["roc_auc"],
                "harm_all": mk("harm_rate", range(1, GROUPS + 1)), "help_all": mk("help_rate", range(1, GROUPS + 1))})
    return out


def evaluate_paper(variant, model, loaders):
    model.eval()
    feat_v, y_v, c_v = collect(model, loaders["val"])
    S_v, C_v, M_v = score_all(variant, model, feat_v, c_v)
    c_v_np = c_v.cpu().numpy()
    pred_v, t_v = predictions(y_v, S_v, S_v, True)
    result = {"val": summarize(y_v, S_v, C_v, pred_v, c_v_np, M_v), "thresholds_val_crossfit": t_v}
    per_case = {"val_y": y_v, "val_score": S_v[0], "val_concepts": C_v[0]}
    if EVALUATE_TEST:
        feat_t, y_t, c_t = collect(model, loaders["test"])
        S_t, C_t, M_t = score_all(variant, model, feat_t, c_t)
        pred_t, t_t = predictions(y_v, S_v, S_t, False)
        result["test"] = summarize(y_t, S_t, C_t, pred_t, c_t.cpu().numpy(), M_t)
        result["thresholds_test_from_val"] = t_t
        per_case.update({"test_y": y_t, "test_score": S_t[0], "test_concepts": C_t[0],
                         "test_pred_k0": pred_t[0], "test_scores_all_k": S_t})
    return result, per_case


# ---- DeLong ghép cặp (Sun & Xu 2014) ----
def _midrank(x):
    order = np.argsort(x, kind="mergesort"); z = x[order]; n = len(x); t = np.zeros(n); i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        t[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n); out[order] = t
    return out


def delong_paired(y, s_a, s_b):
    pos, neg = y == 1, y == 0
    m, n = int(pos.sum()), int(neg.sum())
    aucs, v10, v01 = [], [], []
    for s in (s_a, s_b):
        tx, ty, tz = _midrank(s[pos]), _midrank(s[neg]), _midrank(np.r_[s[pos], s[neg]])
        aucs.append((tz[:m].sum() - m * (m + 1) / 2) / (m * n))
        v10.append((tz[:m] - tx) / n); v01.append(1 - (tz[m:] - ty) / m)
    cov = np.cov(np.vstack(v10)) / m + np.cov(np.vstack(v01)) / n
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = (aucs[1] - aucs[0]) / np.sqrt(max(var, 1e-12))
    return {"auc_A": float(aucs[0]), "auc_D": float(aucs[1]), "delta": float(aucs[1] - aucs[0]),
            "z": float(z), "p": float(2 * stats.norm.sf(abs(z)))}


_y = np.r_[np.zeros(40), np.ones(20)].astype(int); _r = np.random.RandomState(0)
_s1, _s2 = _r.rand(60) + 0.3 * _y, _r.rand(60) + 0.5 * _y
_d = delong_paired(_y, _s1, _s2)
assert abs(_d["auc_A"] - roc_auc_score(_y, _s1)) < 1e-9 and abs(_d["auc_D"] - roc_auc_score(_y, _s2)) < 1e-9
print("✓ Đã định nghĩa cách chấm chung; DeLong khớp sklearn")

## 9. Train và chấm 6 seed
Mỗi seed: train A rồi D (hoặc nạp checkpoint có sẵn), chấm, lưu ngay vào `runs.json` và `per_case_seed{s}.npz`.

In [ ]:
# ===== 9. TRAIN VÀ CHẤM 6 SEED =====
def find_checkpoint(tag, seed):
    for root in (OUT, Path("/kaggle/input")):
        if root.exists():
            hits = sorted(root.rglob(f"ecbm_derm7pt_paper/{tag}_seed_{seed}.pth")) if root != OUT else \
                ([OUT / f"{tag}_seed_{seed}.pth"] if (OUT / f"{tag}_seed_{seed}.pth").exists() else [])
            if hits:
                return hits[0]
    return None


RUNS = json.loads((OUT / "runs.json").read_text()) if (OUT / "runs.json").exists() else {}
TIMING = {}
for s in SEEDS:
    if str(s) in RUNS and ("test" in RUNS[str(s)]["D"] or not EVALUATE_TEST):
        print(f"seed {s}: đã có"); continue
    tr_loader, va_loader, te_loader = get_dataloaders(s)
    loaders = {"val": va_loader, "test": te_loader}
    entry, cases = {}, {}
    for name, tag, make, loss_fn in (("A", "baseline", create_model, compute_loss),
                                     ("D", "gated", create_gated_model, gated_loss)):
        ckpt = find_checkpoint(tag, s)
        if ckpt is not None:
            m = make(); m.load_state_dict(torch.load(ckpt, map_location=DEVICE)); m.eval()
            if ckpt != OUT / f"{tag}_seed_{s}.pth":
                shutil.copy(ckpt, OUT / f"{tag}_seed_{s}.pth")
            epochs = None
            print(f"  nạp {tag} seed {s} từ {ckpt}")
        else:
            m, hist = train_model(s, make, loss_fn, tag)
            epochs = len(hist)
        variants = ("A_joint", "A_author") if name == "A" else ("D",)
        for v in variants:
            t0 = time.time()
            res, pc = evaluate_paper(v, m, loaders)
            res["epochs_trained"] = epochs
            res["eval_seconds"] = time.time() - t0
            entry[v] = res
            cases.update({f"{v}_{k}": val for k, val in pc.items()})
        if name == "D":
            entry["D"]["theta"] = [t.detach().cpu().numpy().tolist() for t in m.theta]
            entry["D"]["gate"] = m.gate(torch.arange(GROUPS + 1, device=DEVICE)).detach().cpu().numpy().tolist()
        del m
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    RUNS[str(s)] = entry
    (OUT / "runs.json").write_text(json.dumps(RUNS, indent=1))
    np.savez_compressed(OUT / f"per_case_seed{s}.npz", **{k: np.asarray(v) for k, v in cases.items()})
    for v in ("A_joint", "A_author", "D"):
        for split in ("val", "test"):
            if split in entry[v]:
                u = entry[v][split]["unintervened"]; r = entry[v][split]
                print(f"seed {s} {v:8s} {split:4s}: AUC {u['roc_auc']:.4f} | F1 {u['f1_macro']:.4f} | concept {u['concept_accuracy']:.4f}"
                      f"/{u['concept_f1_macro']:.4f} | F1 sửa 1–3 {r['f1_k1_3']:.4f} | hại/giúp 1–3 {r['harm_k1_3']:.4f}/"
                      f"{r['help_k1_3']:.4f} | sửa 7 F1/AUC {r['f1_k7']:.4f}/{r['auc_k7']:.4f}", flush=True)

## 10. Kết quả
- **Bảng 1:** chẩn đoán và concept khi chưa can thiệp (trung bình ± độ lệch chuẩn qua 6 seed).
- **Bảng 2:** can thiệp cho A (1 : 1 : 0.01), A (giao thức tác giả) và D.
- **Kiểm định:** DeLong ghép cặp từng seed trên test; t-test ghép cặp theo seed: D so với A (1 : 1 : 0.01) cho chẩn đoán và concept, và D so với A (giao thức tác giả) cho can thiệp.
- **Hình:** đường can thiệp trên test; Δθ của GA-ECBM so với trọng số checklist 7 điểm.

### Cách đọc bảng

| Cột | Ý nghĩa |
|---|---|
| Acc, BAcc | độ chính xác; độ chính xác cân bằng = trung bình của Sens và Spec |
| Macro F1 | trung bình F1 của lớp melanoma và lớp lành |
| Sens, Spec | tỉ lệ bắt đúng melanoma; tỉ lệ nhận đúng ca lành |
| AUC | ROC-AUC trên xác suất melanoma |
| Concept acc | tỉ lệ đoán đúng trạng thái concept, gộp 7 nhóm |
| Concept Macro F1 | Macro F1 theo trạng thái của từng nhóm, lấy trung bình 7 nhóm |
| Đúng cả 7 | tỉ lệ ca đoán đúng cả 7 nhóm concept |
| F1 / AUC sửa 1–3 | trung bình khi sửa 1, 2 và 3 nhóm (mỗi k lấy trung bình qua mọi tập con cùng cỡ) |
| F1 / AUC sửa 7 | khi sửa đủ 7 nhóm |
| Giúp / Gây hại 1–3 | tỉ lệ ca sai → đúng / đúng → sai so với lúc chưa sửa, trung bình k = 1–3 |
| Giúp / Gây hại TB k=1–7 | như trên, trung bình k = 1…7 |
| Δ, tăng ở x/6 seed, p | chênh lệch D − A trung bình qua seed, số seed D cao hơn, p của t-test ghép cặp |
| ρ | tương quan Spearman giữa Δθ (melanoma − lành) và trọng số checklist 7 điểm |

In [ ]:
# ===== 10. BẢNG, KIỂM ĐỊNH, HÌNH =====
import matplotlib.pyplot as plt
seeds = [str(s) for s in SEEDS if str(s) in RUNS]
splits = ["val"] + (["test"] if EVALUATE_TEST else [])
NAMES = {"A_joint": "ECBM", "A_author": "ECBM (can thiệp theo tác giả)", "D": "GA-ECBM"}
ms = lambda vals: f"{np.mean(vals):.3f} ± {np.std(vals, ddof=1):.3f}" if len(vals) > 1 else f"{vals[0]:.3f}"

tables = {}
for split in splits:
    rows = []
    for v in ("A_joint", "D"):
        u = [RUNS[s][v][split]["unintervened"] for s in seeds]
        rows.append({"Mô hình": NAMES[v], **{lab: ms([x[key] for x in u]) for key, lab in (
            ("accuracy", "Acc"), ("balanced_accuracy", "BAcc"), ("f1_macro", "Macro F1"), ("sensitivity", "Sens"),
            ("specificity", "Spec"), ("roc_auc", "AUC"), ("concept_accuracy", "Concept acc"),
            ("concept_f1_macro", "Concept Macro F1"), ("exact_match", "Đúng cả 7"))}})
    t1 = pd.DataFrame(rows); tables[f"bang1_{split}"] = t1
    rows = []
    for v in ("A_joint", "A_author", "D"):
        r = [RUNS[s][v][split] for s in seeds]
        rows.append({"Mô hình": NAMES[v], **{lab: ms([x[key] for x in r]) for key, lab in (
            ("f1_k1_3", "F1 sửa 1–3"), ("auc_k1_3", "AUC sửa 1–3"), ("help_k1_3", "Giúp 1–3"), ("harm_k1_3", "Gây hại 1–3"),
            ("f1_k7", "F1 sửa 7"), ("auc_k7", "AUC sửa 7"), ("help_all", "Giúp TB k=1–7"), ("harm_all", "Gây hại TB k=1–7"))}})
    t2 = pd.DataFrame(rows); tables[f"bang2_{split}"] = t2
    print(f"\n===== {split.upper()} — Bảng 1: chưa can thiệp ({len(seeds)} seed) ====="); print(t1.to_string(index=False))
    print(f"\n===== {split.upper()} — Bảng 2: can thiệp ====="); print(t2.to_string(index=False))
for name, t in tables.items():
    t.to_csv(OUT / f"{name}.csv", index=False)

# Kiểm định
TESTS = {}
for split in splits:
    t = {}
    for key in ("roc_auc", "f1_macro", "balanced_accuracy", "concept_accuracy", "concept_f1_macro"):
        d = [RUNS[s]["D"][split]["unintervened"][key] - RUNS[s]["A_joint"][split]["unintervened"][key] for s in seeds]
        t[key] = {"delta_mean": float(np.mean(d)), "seeds_improved": int(sum(x > 0 for x in d)),
                                 "paired_t_p": float(stats.ttest_1samp(d, 0).pvalue) if len(d) > 1 else None, "per_seed": d}
    for key in ("f1_k1_3", "harm_k1_3", "auc_k7"):
        d = [RUNS[s]["D"][split][key] - RUNS[s]["A_joint"][split][key] for s in seeds]
        t[key] = {"delta_mean": float(np.mean(d)), "seeds_improved": int(sum(x > 0 for x in d)),
                  "paired_t_p": float(stats.ttest_1samp(d, 0).pvalue) if len(d) > 1 else None, "per_seed": d}
    TESTS[split] = t
if EVALUATE_TEST:
    TESTS["delong_test"] = {}
    for s in seeds:
        pc = np.load(OUT / f"per_case_seed{s}.npz")
        TESTS["delong_test"][s] = delong_paired(pc["A_joint_test_y"], pc["A_joint_test_score"], pc["D_test_score"])
    print("\nDeLong ghép cặp trên test (D so với A):")
    for s, r in TESTS["delong_test"].items():
        print(f"  seed {s}: AUC {r['auc_A']:.4f} → {r['auc_D']:.4f} (Δ {r['delta']:+.4f}), p = {r['p']:.4f}")
for split in splits:
    print(f"\nT-test ghép cặp theo seed ({split}):")
    for key, r in TESTS[split].items():
        print(f"  Δ{key}: {r['delta_mean']:+.4f} | tăng ở {r['seeds_improved']}/{len(seeds)} seed | p = {r['paired_t_p']}")

# Can thiệp: GA-ECBM so với ECBM can thiệp theo giao thức tác giả (phép so sánh chính về can thiệp)
TESTS_AUTHOR = {}
for split in splits:
    t = {}
    for key in ("f1_k1_3", "auc_k1_3", "help_k1_3", "harm_k1_3", "f1_k7", "auc_k7", "help_all", "harm_all"):
        d = [RUNS[s]["D"][split][key] - RUNS[s]["A_author"][split][key] for s in seeds]
        t[key] = {"delta_mean": float(np.mean(d)), "seeds_D_higher": int(sum(x > 0 for x in d)),
                  "paired_t_p": float(stats.ttest_1samp(d, 0).pvalue) if len(d) > 1 else None, "per_seed": d}
    TESTS_AUTHOR[split] = t
    print(f"\nT-test ghép cặp theo seed ({split}), D so với ECBM can thiệp theo giao thức tác giả:")
    for key, r in t.items():
        p_txt = "—" if r["paired_t_p"] is None else f"{r['paired_t_p']:.4f}"
        print(f"  Δ{key}: {r['delta_mean']:+.4f} | D cao hơn ở {r['seeds_D_higher']}/{len(seeds)} seed | p = {p_txt}")

# Hình 1: đường can thiệp
split = "test" if EVALUATE_TEST else "val"
COL = {"A_joint": "#2a78d6", "A_author": "#8e44ad", "D": "#eb6834"}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
for v in ("A_joint", "A_author", "D"):
    for ax, key in zip(axes, ("f1_macro", "roc_auc", "harm_rate")):
        curves = []
        for s in seeds:
            r = RUNS[s][v][split]
            k0 = r["unintervened"][key] if key != "harm_rate" else 0.0
            curves.append([k0] + [r["intervention_by_k"][str(k)][key] for k in range(1, GROUPS + 1)])
        c = np.array(curves); mu, sd = c.mean(0), c.std(0, ddof=1) if len(c) > 1 else np.zeros(c.shape[1])
        ax.plot(range(GROUPS + 1), mu, marker="o", ms=4, lw=2, color=COL[v], label=NAMES[v])
        ax.fill_between(range(GROUPS + 1), mu - sd, mu + sd, color=COL[v], alpha=0.15, lw=0)
for ax, title in zip(axes, ("Macro F1", "ROC-AUC", "Tỉ lệ gây hại")):
    ax.set_title(f"{title} ({split})", loc="left"); ax.set_xlabel("số nhóm concept được sửa (k)")
    ax.grid(color="#e4e3df", lw=0.8); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, fontsize=9)
fig.tight_layout(); fig.savefig(OUT / f"intervention_curves_{split}.png", dpi=150); plt.show()

# Hình 2: θ của GA-ECBM so với checklist 7 điểm
SEVEN_POINT = {("pigment_network", "atypical"): 2, ("blue_whitish_veil", "present"): 2,
               ("vascular_structures", "linear irregular"): 2, ("vascular_structures", "dotted"): 2,
               ("pigmentation", "diffuse irregular"): 1, ("pigmentation", "localized irregular"): 1,
               ("streaks", "irregular"): 1, ("dots_and_globules", "irregular"): 1,
               ("regression_structures", "blue areas"): 1, ("regression_structures", "white areas"): 1,
               ("regression_structures", "combinations"): 1}
inv = {name: {i: lab for lab, i in LABEL_MAPPING[name].items()} for name in CONCEPT_NAMES}
def checklist_score(c):
    return sum(SEVEN_POINT.get((name, inv[name][int(c[g])]), 0) for g, name in enumerate(CONCEPT_NAMES))
if "seven_point_score" in test_df.columns:
    truth = np.array([[LABEL_MAPPING[n].get(str(r[n]).strip().lower(), 0) for n in CONCEPT_NAMES] for _, r in val_df.iterrows()])
    agree = float(np.mean([checklist_score(c) == int(v) for c, v in zip(truth, val_df["seven_point_score"])]))
    print(f"\nKiểm tra bảng trọng số checklist: điểm tính từ concept khớp cột seven_point_score ở {agree:.1%} ca validation")
labels, dtheta, weights = [], [], []
for g, name in enumerate(CONCEPT_NAMES):
    th = np.mean([np.array(RUNS[s]["D"]["theta"][g]) for s in seeds], axis=0)      # [2, K_g]
    d = (th[1] - th[0]); d = d - d[0]                                               # melanoma − lành, so với 'absent'
    for i in range(1, CONCEPT_NUM_CLASSES[name]):
        labels.append(f"{name}: {inv[name][i]}"); dtheta.append(float(d[i])); weights.append(SEVEN_POINT.get((name, inv[name][i]), 0))
rho = stats.spearmanr(dtheta, weights)
print(f"Tương quan Spearman giữa Δθ (melanoma − lành) và trọng số checklist 7 điểm: ρ = {rho.correlation:.3f}, p = {rho.pvalue:.4f}")
order = np.argsort(dtheta)
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(range(len(order)), [dtheta[i] for i in order], color=["#eb6834" if weights[i] == 2 else "#f2b48f" if weights[i] == 1 else "#b8b8b0" for i in order])
ax.set_yticks(range(len(order))); ax.set_yticklabels([labels[i] for i in order], fontsize=8)
ax.axvline(0, color="#5f5e5a", lw=0.8)
ax.set_title(f"Δθ của GA-ECBM (melanoma − lành, so với 'absent'), TB {len(seeds)} seed\n"
             f"màu: trọng số checklist 7 điểm (đậm = 2, nhạt = 1, xám = 0); Spearman ρ = {rho.correlation:.2f}", loc="left", fontsize=10)
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); fig.savefig(OUT / "theta_vs_seven_point.png", dpi=150); plt.show()

summary = {"seeds": SEEDS, "test_evaluated": EVALUATE_TEST, "tables": {k: v.to_dict(orient="records") for k, v in tables.items()},
           "tests": TESTS, "tests_D_vs_A_author": TESTS_AUTHOR, "theta_checklist_spearman": {"rho": float(rho.correlation), "p": float(rho.pvalue)},
           "protocol": {"threshold": "theo k, tối đa Macro F1; val: chọn chéo hai nửa val; test: chọn trên toàn val",
                        "intervention": "mọi tập con của 7 nhóm (127)", "A_author_round2_weight": AUTHOR_ROUND2_WEIGHT}}
(OUT / "summary.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False, default=float))
print("\nĐã lưu", OUT, "| file:", sorted(p.name for p in OUT.iterdir() if not p.name.endswith(".pth")))